# HRM Training from Scratch: 6×6 Sudoku

This notebook trains the HRM (Hierarchical Reasoning Model) **from scratch** on 6×6 Sudoku puzzles.

**Why from scratch?** The pre-trained HRM checkpoint was trained on 9×9 puzzles (vocab_size=11, seq_len=81). The 6×6 task uses different dimensions (vocab_size=8, seq_len=36), so the embedding and output layers are incompatible.

**Key Settings (optimized for Kaggle T4 GPU, 12h limit):**
- Dataset: 1000 train + 100 test (self-generated, fast)
- Full training (no LoRA — all 27M params)
- Epochs: 5000, eval every 1000 epochs
- Batch size: 64 (6×6 puzzles are much smaller than 9×9)

**Estimated Runtime: ~2-4 hours**

## 1. Setup Environment

In [ ]:
!git clone https://github.com/jagan25-mj/NHRT.git
%cd NHRT
!pip install setuptools wheel
!pip install einops tqdm pydantic wandb omegaconf hydra-core huggingface_hub peft argdantic coolname --quiet


## 2. Create the 6×6 Sudoku Dataset Builder

The base HRM repo doesn't include a 6×6 dataset builder, so we create one.

In [ ]:
%%writefile dataset/build_6x6_sudoku_dataset.py
import os
import json
import random
import numpy as np
from tqdm import tqdm
import argparse

try:
    from dataset.common import PuzzleDatasetMetadata
except ImportError:
    from common import PuzzleDatasetMetadata

def is_valid(board, r, c, val):
    if val in board[r]: return False
    if val in board[:, c]: return False
    br, bc = 2 * (r // 2), 3 * (c // 3)
    if val in board[br:br+2, bc:bc+3]: return False
    return True

def solve(board):
    for r in range(6):
        for c in range(6):
            if board[r, c] == 0:
                nums = list(range(1, 7))
                random.shuffle(nums)
                for val in nums:
                    if is_valid(board, r, c, val):
                        board[r, c] = val
                        if solve(board): return True
                        board[r, c] = 0
                return False
    return True

def generate_6x6_board():
    board = np.zeros((6, 6), dtype=int)
    solve(board)
    return board

def generate_puzzles(num_puzzles, num_holes):
    inputs, labels = [], []
    for _ in tqdm(range(num_puzzles), desc="Generating 6x6 puzzles"):
        solution = generate_6x6_board()
        puzzle = solution.copy()
        cells = [(r, c) for r in range(6) for c in range(6)]
        random.shuffle(cells)
        for r, c in cells[:num_holes]:
            puzzle[r, c] = 0
        inputs.append(puzzle)
        labels.append(solution)
    return inputs, labels

def convert_subset(set_name, output_dir, num_train, num_test, holes):
    num_puzzles = num_train if set_name == "train" else num_test
    inputs, labels = generate_puzzles(num_puzzles, holes)
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    for i, (inp, out) in enumerate(zip(inputs, labels)):
        results["inputs"].append(inp.flatten() + 1)
        results["labels"].append(out.flatten() + 1)
        results["puzzle_indices"].append(i + 1)
        results["puzzle_identifiers"].append(0)
        results["group_indices"].append(i + 1)
    results = {
        "inputs": np.array(results["inputs"], dtype=np.int32),
        "labels": np.array(results["labels"], dtype=np.int32),
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }
    metadata = PuzzleDatasetMetadata(
        seq_len=36, vocab_size=8, pad_id=0, ignore_label_id=0,
        blank_identifier_id=1, num_puzzle_identifiers=1,
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1, sets=["all"]
    )
    save_dir = os.path.join(output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
    with open(os.path.join(output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", type=str, default="data/sudoku-6x6")
    parser.add_argument("--num-train", type=int, default=1000)
    parser.add_argument("--num-test", type=int, default=100)
    parser.add_argument("--holes", type=int, default=20)
    args = parser.parse_args()
    
    convert_subset("train", args.output_dir, args.num_train, args.num_test, args.holes)
    convert_subset("test", args.output_dir, args.num_train, args.num_test, args.holes)


## 3. Generate 6×6 Dataset

In [ ]:
!python dataset/build_6x6_sudoku_dataset.py --output-dir data/sudoku-6x6 --num-train 1000 --num-test 100 --holes 20

import os
print("\nDataset created:")
for split in ["train", "test"]:
    split_dir = f"data/sudoku-6x6/{split}"
    if os.path.exists(split_dir):
        files = os.listdir(split_dir)
        print(f"  {split}: {files}\n")


## 4. Train HRM from Scratch on 6×6 Sudoku

Using the standard `pretrain.py` with settings tuned for 6×6:
- **epochs=5000**: Enough iterations for the small model to converge
- **eval_interval=1000**: Check accuracy every 1000 epochs
- **global_batch_size=64**: Larger batches since 6×6 puzzles are small
- **lr=7e-5**: Learning rate from the paper's Sudoku training recipe

In [ ]:
!WANDB_MODE=disabled python pretrain.py \
    data_path=data/sudoku-6x6 \
    epochs=5000 \
    eval_interval=1000 \
    checkpoint_every_eval=True \
    global_batch_size=64 \
    lr=7e-5 \
    puzzle_emb_lr=7e-5 \
    weight_decay=1.0 \
    puzzle_emb_weight_decay=1.0

## 5. Evaluate the Trained Model

In [ ]:
import os, glob, yaml
import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
import numpy as np

os.environ["DISABLE_COMPILE"] = "1"

from pretrain import PretrainConfig, init_train_state, create_dataloader
from models.losses import IGNORE_LABEL_ID

# Find latest checkpoint
latest_step = -1
best_path = None
for root, dirs, files in os.walk("checkpoints"):
    for f in files:
        if f.startswith("step_") and "all_preds" not in f:
            try:
                step = int(f.split("_")[1])
                if step > latest_step:
                    latest_step = step
                    best_path = os.path.join(root, f)
            except: pass

if not best_path:
    print("ERROR: No checkpoint found!")
else:
    print(f"Checkpoint: {best_path} (step {latest_step})")
    
    with open(os.path.join(os.path.dirname(best_path), "all_config.yaml"), "r") as f:
        config = PretrainConfig(**yaml.safe_load(f))
    config.eval_save_outputs = ["logits"]
    config.checkpoint_path = os.path.dirname(best_path)
    
    eval_loader, eval_metadata = create_dataloader(config, "test", test_set_mode=True,
        epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=0, world_size=1)
    
    train_state = init_train_state(config, eval_metadata, world_size=1)
    
    try:
        train_state.model.load_state_dict(
            torch.load(best_path, map_location="cuda", weights_only=True), assign=True)
    except:
        train_state.model.load_state_dict(
            {k.removeprefix("_orig_mod."): v for k, v in 
             torch.load(best_path, map_location="cuda", weights_only=True).items()}, assign=True)
    
    train_state.model = train_state.model.float().cuda()
    train_state.model.eval()
    
    total_correct = 0
    total_cells = 0
    total_exact = 0
    total_puzzles = 0
    
    print("\nRunning evaluation...")
    
    with torch.no_grad():
        for set_name, batch, global_batch_size in eval_loader:
            batch = {k: v.cuda().float() if v.is_floating_point() else v.cuda() for k, v in batch.items()}
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)
            for step_i in range(16):
                carry, _, metrics, preds, all_finish = train_state.model(
                    carry=carry, batch=batch, return_keys=["logits"])
                if all_finish: break
            
            labels = batch["labels"]
            logits = preds.get("logits", None)
            
            if logits is not None:
                predictions = torch.argmax(logits, dim=-1)
                mask = labels != IGNORE_LABEL_ID
                correct = (predictions == labels) & mask
                total_correct += correct.sum().item()
                total_cells += mask.sum().item()
                for i in range(labels.shape[0]):
                    puzzle_mask = mask[i]
                    if puzzle_mask.sum() > 0:
                        total_puzzles += 1
                        if correct[i].sum() == puzzle_mask.sum():
                            total_exact += 1
    
    print("\n" + "=" * 60)
    print(f"6x6 SUDOKU EVALUATION RESULTS (step {latest_step})")
    print("=" * 60)
    print(f"Cell Accuracy:    {total_correct}/{total_cells} = {100*total_correct/max(total_cells,1):.2f}%")
    print(f"Puzzle Accuracy:  {total_exact}/{total_puzzles} = {100*total_exact/max(total_puzzles,1):.2f}%")
    print("=" * 60)

## 6. Final Summary

In [ ]:
import os

print("=" * 60)
print("6x6 SUDOKU TRAINING FROM SCRATCH COMPLETE")
print("=" * 60)

print("\nSaved checkpoints:")
for root, dirs, files in os.walk("checkpoints"):
    for f in files:
        if f.startswith("step_") and "all_preds" not in f:
            full_path = os.path.join(root, f)
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"  {full_path} ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
print("KEY RESULTS TO NOTE:")
print("1. Check training logs for eval results at each 1000-epoch interval")
print("2. Check the 'EVALUATION RESULTS' section above for final accuracy")
print("=" * 60)